In [ ]:
import pandas as pd

df = pd.read_csv("raw_gesture_data.csv", header=None) # raw_gesture_data.csv has no headers
df.head()
# load csv data into a dataframe and then read the top to see of column titles are correct

,0,1,2,3,4,5,6,7,8,9,...,33,34,35,36,37,38,39,40,41,42
0,0.680312,0.684303,0.575322,0.647014,0.489377,0.550064,0.454866,0.434962,0.479575,0.351520,...,0.496096,0.726956,0.428301,0.705024,0.359292,0.698973,0.442338,0.706247,0.479626,closed_fist
1,0.673250,0.691755,0.569266,0.644218,0.500157,0.541688,0.479538,0.437154,0.493842,0.358032,...,0.511469,0.735149,0.440129,0.700302,0.364683,0.688247,0.451864,0.704398,0.496898,closed_fist
2,0.706273,0.660627,0.599902,0.656460,0.490091,0.583547,0.426930,0.483573,0.445228,0.401403,...,0.489710,0.693541,0.407893,0.676620,0.345559,0.689192,0.419823,0.693521,0.459553,closed_fist
3,0.670289,0.675614,0.595504,0.610084,0.564483,0.536124,0.548198,0.481891,0.551295,0.423888,...,0.516887,0.701231,0.424616,0.610621,0.423749,0.609857,0.487289,0.643136,0.508830,closed_fist
4,0.659017,0.684158,0.569728,0.659741,0.479908,0.579726,0.442574,0.482290,0.486319,0.429055,...,0.528514,0.685477,0.450176,0.655698,0.400648,0.654498,0.477924,0.665651,0.509048,closed_fist


In [42]:
# sanity check to verify this matches my csv data
print(df.shape) # 967 samples and 21 different landmarks (x and y for each) plus last column for label = 43

(967, 43)


In [ ]:
# grab my feature set for training, cut out label column
X = df.iloc[:, :-1].values
# grab the column of labels; need them for encoding 
y = df.iloc[:, -1].values

In [44]:
# encode the gesture names to integers

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y = le.fit_transform(y) # match each label to integer value

print(le.classes_)  # view the labels in the order they are encoded

['closed_fist' 'open_palm' 'point_up']


In [45]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,      # 80% of data is going to be used for training, 20% for testing the model
    random_state=42,    # keep same rows of data in training and testing, uses this random state each time 
    stratify=y          # keep same percentage for each gesture in each training and testing set (representative distribution)
)

In [46]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

# Sequential is chosen bc this is standard classification problem (simple feedforward network)
model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train.shape[1],)),     # 128 neurons = balanced default; relu = learn nonlinear patterns
    Dropout(0.3),                                                       # turn off 30% of neurons randomly during training = prevent overfitting

    Dense(64, activation='relu'),
    Dropout(0.3),

    Dense(len(np.unique(y)), activation='softmax')  # number of neurons = number of gestures; softmax converts ouputs into proba. that sum to 1
])

In [48]:
# setup rules for training

model.compile(
    optimizer='adam',       # Adaptive Moment Estimation = common optimizer
    loss='sparse_categorical_crossentropy',     # measure how wrong the model is, compares the softmax output against correct label
    metrics=['accuracy']    # report accuracy each epoch
)

In [49]:
# Training phase = model predicts from X_train data, compares to (y_train) labels, and the updates the weights 
# Validation phase = model predicts on unseen data (X_test) and then compares to (y_test) labels
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),   # unseen data and corresponding labels
    epochs=20,                          # model sees full training set 20 times
    batch_size=32                       # split whole training data into many batches of 32 samples, run each batch through the network and update the weights
    # in my case, I'm getting 967 / 32 =~ 31 batches meaning I'll get about 31 weight updates per epoch
)

# Each epoch = make predictions, compare to 

Epoch 1/20
25/25 [==============================] - 1s 14ms/step - loss: 1.0995 - accuracy: 0.3894 - val_loss: 1.0419 - val_accuracy: 0.6753
Epoch 2/20
25/25 [==============================] - 0s 5ms/step - loss: 1.0375 - accuracy: 0.4515 - val_loss: 1.0058 - val_accuracy: 0.3866
Epoch 3/20
25/25 [==============================] - 0s 5ms/step - loss: 0.9910 - accuracy: 0.5433 - val_loss: 0.9631 - val_accuracy: 0.5670
Epoch 4/20
25/25 [==============================] - 0s 5ms/step - loss: 0.9347 - accuracy: 0.5757 - val_loss: 0.8587 - val_accuracy: 0.8144
Epoch 5/20
25/25 [==============================] - 0s 5ms/step - loss: 0.8556 - accuracy: 0.6831 - val_loss: 0.7623 - val_accuracy: 0.8660
Epoch 6/20
25/25 [==============================] - 0s 5ms/step - loss: 0.7740 - accuracy: 0.7180 - val_loss: 0.6982 - val_accuracy: 0.7887
Epoch 7/20
25/25 [==============================] - 0s 4ms/step - loss: 0.6443 - accuracy: 0.8266 - val_loss: 0.5526 - val_accuracy: 0.9588
Epoch 8/20
25/25 [=

In [ ]:
# test trained model on test dataset (final standalone test)
loss, acc = model.evaluate(X_test, y_test)  # loss = how wrong model is on unseen data, acc = fraction of correct predictions
print("Test accuracy:", acc)

7/7 [==============================] - 0s 2ms/step - loss: 0.0867 - accuracy: 0.9897
Test accuracy: 0.9896907210350037


In [ ]:
# Main idea is to compare predictions to true labels from dataset 

# run trained model on testdata, get softmax output for each sample; ex: [ 0.05, .90, 0.05]
pred = model.predict(X_test)
# get index of hightest proba for each sample (corresponds to label number) ex: [ 0.05, .90, 0.05] -> 1
pred_classes = np.argmax(pred, axis=1)
# takes all the indexes and converts them back to my original gesture labels
pred_labels = le.inverse_transform(pred_classes)

7/7 [==============================] - 0s 2ms/step


In [ ]:
# save model for use in live program

model.save("gesture_model.keras")   # newer, preferred format
model.save("gesture_model.h5")      # legacy format

c:\Users\krhod\OneDrive\Documents\Hand Gesture Recognition\hand-gesture-recognition-modern\gesture-env\lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [ ]:
# library for saving and loading python objects
import joblib

# le is instance of encoder thats been wrapped around the label column from the data; save the mappigns between gesture and numerical labels
joblib.dump(le, "label_encoder.pkl")

['label_encoder.pkl']